## Amostra ampliada: redes dos participantes adicionais

Gera as redes da amostra ampliada **sem alterar** a amostra principal (54 participantes, `data/intermediate` e `data/processed`).

**Critério de inclusão ampliado** (em relação ao critério estrito):
- campo vazio nos questionários é tratado como "sem informação" (não exclui);
- BSL-23 comportamental ≤ 1 (em vez de = 0).

Os demais critérios são mantidos: drogas negativo, não fumante, AUDIT < 8, Hamilton ≤ 7, SKID sem diagnóstico, sem 2º diagnóstico e sem comentário na avaliação SKID.

Os parâmetros de rede são os mesmos de `00_redes/01_build_networks` (8 s, janela de 100 ms, atraso de até 24 ms nos dois sentidos, limiar p99). Saídas em `data/expanded/`; o dataset final usa os mesmos 53 eletrodos da amostra principal.

### Imports e parâmetros

In [5]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mne
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.graph.aggregate import (
    aggregate_static_network,
    clustering_over_time,
    degree_over_time,
    edges_per_time,
    temporal_statistics,
    window_node_features,
)
from src.graph.eeg_crop import crop_continuous_segment
from src.graph.export_networks import save_network_exports
from src.graph.motifs import trans_motifs
from src.graph.network import graph_metrics
from src.graph.threshold import estimate_threshold
from src.graph.tvg import build_tvg_parallel

DATA = Path("../../data")
SOURCE_DIR = DATA / "preprocessed"
METADATA_FILE = DATA / "metadata" / "participants.csv"
MAIN_DATASET = DATA / "processed" / "dataset_features.parquet"
MAIN_ALIGNED_DIR = DATA / "intermediate_aligned"

EXP_DIR = DATA / "expanded"
OUTPUT_DIR = EXP_DIR / "intermediate"
NETWORKS_DIR = OUTPUT_DIR / "networks"
WINDOWS_DIR = OUTPUT_DIR / "windows"
EXP_DATASET = EXP_DIR / "dataset_features.parquet"
for d in (OUTPUT_DIR, NETWORKS_DIR, WINDOWS_DIR):
    d.mkdir(parents=True, exist_ok=True)

WINDOW_MS = 100
MAX_LAG_MS = 25
THRESHOLD_PERCENTILE = 99.0

_common = json.loads((DATA / "intermediate" / "tvg_common_continuous_summary.json").read_text(encoding="utf-8"))
TVG_T0 = float(_common["t0"])
TVG_T1 = float(_common["t1"])
N_SEC = int(_common.get("N_SEC_sugerido", 8))
print(f"Recorte: {N_SEC} s dentro de [{TVG_T0:.3f}, {TVG_T1:.3f}] s")

Recorte: 8 s dentro de [60.002, 68.274] s


### Seleção da amostra ampliada

In [6]:
RENAME = {
    "ID": "subject_id",
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status",
}


def expanded_mask(m: pd.DataFrame) -> pd.Series:
    skid = m["skid_diagnoses"].astype(str).str.strip().str.lower()
    return (
        ((m["drug"] == 0) | m["drug"].isna())
        & ((m["smoking"] == 1) | m["smoking"].isna())
        & ((m["audit"] < 8) | m["audit"].isna())
        & ((m["hamilton_scale"] <= 7) | m["hamilton_scale"].isna())
        & ((m["bsl23_behavior"] <= 1) | m["bsl23_behavior"].isna())
        & ((skid == "none") | m["skid_diagnoses"].isna())
        & m["skid_diagnoses_2"].isna()
        & m["comments_skid_assessment"].isna()
    )


meta = pd.read_csv(METADATA_FILE).rename(columns=RENAME)
eeg_files = {f.stem: f for f in SOURCE_DIR.glob("sub-*.set")}


def files_for(subject: str) -> list[Path]:
    eo = eeg_files.get(f"{subject}_EO") or eeg_files.get(f"{subject}_E0")
    ec = eeg_files.get(f"{subject}_EC")
    return [f for f in (eo, ec) if f is not None]


has_eeg = meta["subject_id"].map(lambda s: len(files_for(s)) == 2)
expanded_ids = set(meta.loc[expanded_mask(meta) & has_eeg, "subject_id"])
main_ids = set(pd.read_parquet(MAIN_DATASET, columns=["subject_id"])["subject_id"])
assert main_ids <= expanded_ids, "A amostra principal deveria estar contida na ampliada"
new_ids = sorted(expanded_ids - main_ids)

age = meta.set_index("subject_id")["age"]
older = lambda ids: sum(age[s] >= "55" for s in ids)
print(f"Amostra principal: {len(main_ids)} ({len(main_ids) - older(main_ids)} jovens, {older(main_ids)} mais velhos)")
print(f"Amostra ampliada: {len(expanded_ids)} ({len(expanded_ids) - older(expanded_ids)} jovens, {older(expanded_ids)} mais velhos)")
print(f"Participantes adicionais: {len(new_ids)} ({len(new_ids) - older(new_ids)} jovens, {older(new_ids)} mais velhos)")
new_files = [f for s in new_ids for f in files_for(s)]
print(f"Registros a processar: {len(new_files)}")

Amostra principal: 54 (35 jovens, 19 mais velhos)
Amostra ampliada: 69 (46 jovens, 23 mais velhos)
Participantes adicionais: 15 (11 jovens, 4 mais velhos)
Registros a processar: 30


### Redes dos participantes adicionais

Mesmo fluxo de `00_redes/01_build_networks`: recorte → limiar nulo (p99) → motifs → TVG → métricas. Registros já processados são pulados.

In [7]:
for file in tqdm(new_files, desc="Registros"):
    subject_id, condition = file.stem.split("_")
    stem = f"{subject_id}_{condition}"
    out_file = OUTPUT_DIR / f"{stem}.parquet"
    if out_file.exists() and (NETWORKS_DIR / f"{stem}_nodes.csv").exists():
        continue

    raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
    raw.pick(picks="eeg")
    ch_names = list(raw.ch_names)
    eeg, sfreq, _, _ = crop_continuous_segment(raw, TVG_T0, TVG_T1, N_SEC, file_name=file.name)

    window_size = int(WINDOW_MS / 1000 * sfreq)
    max_lag = int(MAX_LAG_MS / 1000 * sfreq)
    threshold_info = estimate_threshold(eeg, window_size, max_lag, percentile=THRESHOLD_PERCENTILE)
    threshold = threshold_info["threshold"]
    tvg = build_tvg_parallel(
        motifs=trans_motifs(eeg), window_size=window_size, max_lag=max_lag, threshold=threshold
    )

    rea = aggregate_static_network(tvg)
    edges_stats = temporal_statistics(edges_per_time(tvg))
    net = graph_metrics(rea, n_windows=tvg.shape[2])
    win = window_node_features(tvg)

    features = {}
    for metric in ("degree", "clustering", "betweenness"):
        for i, v in enumerate(net[metric]):
            features[f"{metric}_{i}"] = v
    for metric, values in win.items():
        for i, v in enumerate(values):
            features[f"{metric}_{i}"] = v
    features["sync_threshold"] = threshold
    features["sync_threshold_null_mean"] = threshold_info["null_mean"]
    features["sync_threshold_null_std"] = threshold_info["null_std"]
    features["edges_mean"] = edges_stats["mean"]
    features["edges_std"] = edges_stats["std"]
    features["edges_cv"] = edges_stats["cv"]

    df = pd.DataFrame(features, index=[0])
    df["subject_id"] = subject_id
    df["condition"] = condition
    df.to_parquet(out_file, index=False)
    np.savez_compressed(
        WINDOWS_DIR / f"{stem}.npz",
        degree=degree_over_time(tvg).astype(np.int16),
        clustering=clustering_over_time(tvg).astype(np.float32),
        ch_names=np.array(ch_names),
    )
    save_network_exports(tvg=tvg, rea=rea, ch_names=ch_names, networks_dir=NETWORKS_DIR, stem=stem)
    print(f"{stem}: limiar={threshold:.4f} | janelas={tvg.shape[2]} | eletrodos={len(ch_names)}")

print("Concluído.")

Registros: 100%|██████████| 30/30 [00:00<00:00, 4865.41it/s]

Concluído.


### Dataset da amostra ampliada

Os 54 participantes da amostra principal reaproveitam os arquivos já alinhados (`data/intermediate_aligned`); os adicionais são realinhados por nome de eletrodo, nos mesmos 53 eletrodos.

In [8]:
NODE_METRIC_PREFIXES = (
    "degree_", "clustering_", "betweenness_", "hub_frequency_", "sd_degree_", "bin_clustering_",
)


def remap_row(features: pd.DataFrame, elects: list[str]) -> dict:
    row = features.iloc[0].to_dict()
    out = {}
    for k, v in row.items():
        prefix = next((p for p in NODE_METRIC_PREFIXES if k.startswith(p) and k[len(p):].isdigit()), None)
        if prefix is None:
            out[k] = v
        else:
            out[f"{prefix}{elects[int(k[len(prefix):])]}"] = v
    return out


main_frames = [
    pd.read_parquet(MAIN_ALIGNED_DIR / f"{s}_{c}.parquet") for s in sorted(main_ids) for c in ("EO", "EC")
]
node_cols = [c for c in main_frames[0].columns if c.startswith(NODE_METRIC_PREFIXES)]

new_frames = []
for path in sorted(OUTPUT_DIR.glob("sub-*.parquet")):
    subject_id, condition = path.stem.split("_")
    if subject_id not in new_ids:
        continue
    nodes = pd.read_csv(NETWORKS_DIR / f"{path.stem}_nodes.csv")
    row = remap_row(pd.read_parquet(path), nodes["Elect"].astype(str).str.strip().tolist())
    row["subject_id"] = subject_id
    row["condition"] = condition.replace("E0", "EO")
    new_frames.append(pd.DataFrame([row]))

df_features = pd.concat(main_frames + new_frames, ignore_index=True)
other_cols = [c for c in df_features.columns if not c.startswith(NODE_METRIC_PREFIXES)]
df_features = df_features[node_cols + other_cols]
print(f"Registros: {len(df_features)} | colunas por eletrodo: {len(node_cols)}")

df_final = df_features.merge(meta, on="subject_id", how="left", validate="m:1")
for col in ("relationship_status", "handedness", "education"):
    df_final[col] = df_final[col].str.lower()
df_final = df_final[df_final["subject_id"].isin(expanded_ids)].copy()

meta_cols = ["age", "gender", "handedness", "education", "relationship_status"]
df_meta = pd.get_dummies(df_final[meta_cols].copy(), columns=meta_cols, drop_first=True)
df_analysis = pd.concat([df_final, df_meta], axis=1)
df_analysis.to_parquet(EXP_DATASET, index=False)

counts = df_analysis.drop_duplicates("subject_id")["age"].map(lambda a: "mais velhos" if a >= "55" else "jovens")
print(f"Salvo: {EXP_DATASET} | {df_analysis['subject_id'].nunique()} participantes, {len(df_analysis)} registros")
print(counts.value_counts().to_string())

C:\Users\Anne\AppData\Local\Temp\ipykernel_53552\2003046034.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_features = pd.concat(main_frames + new_frames, ignore_index=True)


Registros: 138 | colunas por eletrodo: 318
Salvo: ..\..\data\expanded\dataset_features.parquet | 69 participantes, 138 registros
age
jovens         46
mais velhos    23
